# Bài 1: Tìm đường đi ngắn nhất từ 2 đỉnh trong đồ thị

## 1. Giới thiệu

OPEN: tập các trạng thái đã được sinh ra nhưng chưa được xét đến.
CLOSED: tập các trạng thái đã được xét đến.
COST(p, q): là khoảng cách giữa p, q.
g(p): khoảng cách từ trạng thái đầu đến trạng thái hiện tại p.
path(p): trạng thái trước đó để đi tới trạng thái p
Bước 1:
Open = {s}
Close = {}
Bước 2: while (Open !={})
+ Chọn trạng thái (đỉnh) tốt nhất p theo g(p) trong OPEN (xóa p khỏi OPEN).
+ Chuyển p qua CLOSED
+ Xét các đỉnh q kề với p:
* Nếu q chưa có trong OPEN và CLOSED:
- g(q) = g(p) + cost(p, q)
- path(q) = p
- Thêm q vào OPEN
* Nếu q đã có trong OPEN và g(q) > g(p) + cost(p, q):
- g(q) = g(p) + cost(p, q)
- path(q) = p
* Nếu q có trong CLOSED: không làm gì cả
Bước 3: Truy hồi trạng thái đích.

---
## 2. Bài toán

Tìm đường đi ngắn nhất từ đỉnh A đến đỉnh B.



In [1]:
import heapq
import pprint

# =========================
# MinHeap: Hỗ trợ hàng đợi ưu tiên
# =========================
class MinHeap:
    def __init__(self):
        self.items = []

    def empty(self):
        return len(self.items) == 0

    def push(self, item):
        heapq.heappush(self.items, item)

    def pop(self):
        return heapq.heappop(self.items)

    def check(self, item):
        return item in self.items

    def update(self, old_item, new_item):
        for i in range(len(self.items)):
            if self.items[i] == old_item:
                self.items[i] = new_item
                heapq.heapify(self.items)
                break

In [2]:
# =========================
# Biểu diễn đồ thị bằng danh sách kề có trọng số
# =========================
COST = {
    'A': {'C': 9, 'D': 7, 'E': 13, 'F': 20},
    'B': {},
    'C': {'H': 6},
    'D': {'E': 4, 'H': 8},
    'E': {'I': 3, 'K': 4},
    'F': {'G': 4, 'I': 6},
    'G': {},
    'H': {'K': 5},
    'I': {'B': 5, 'K': 9},
    'K': {'B': 6}
}

# =========================
# Heuristic h(n) cho A* (Khoảng cách ước lượng đến đích B)
# =========================
h = {
    'A': 14, 'B': 0, 'C': 15, 'D': 6, 'E': 8,
    'F': 7, 'G': 12, 'H': 10, 'I': 4, 'K': 2
}

def LayKe(G, a):
    if G.get(a) is None:
        return None
    return list(G[a].keys())

def find_path(path, start, goal):
    result = []
    v = goal
    while v is not None:
        result.append(v)
        v = path.get(v)
    result.reverse()
    return result

In [3]:
def BestFirstSearch(G, start, goal):
    if G.get(start) is None or G.get(goal) is None:
        return (None, None)

    path = {}
    g = {}
    OPEN = MinHeap()
    CLOSED = []

    OPEN.push((0, start))
    path[start] = None
    g[start] = 0

    while not OPEN.empty():
        cost_u, u = OPEN.pop()
        if u in CLOSED: continue
        CLOSED.append(u)

        if u == goal: break

        for v in G[u]:
            new_cost = g[u] + G[u][v]
            if g.get(v) is None:
                g[v] = new_cost
                path[v] = u
                OPEN.push((g[v], v))
            elif new_cost < g[v]:
                old_item = (g[v], v)
                g[v] = new_cost
                path[v] = u
                OPEN.update(old_item, (g[v], v))
    return (path, g)

In [4]:
def AStartSearch(G, start, goal, h):
    if G.get(start) is None or G.get(goal) is None:
        return (None, None)

    path = {}
    g = {}
    f = {}
    OPEN = MinHeap()
    CLOSED = []

    path[start] = None
    g[start] = 0
    f[start] = g[start] + h.get(start, 0)
    OPEN.push((f[start], start))

    while not OPEN.empty():
        _, p = OPEN.pop()
        if p in CLOSED: continue
        CLOSED.append(p)

        if p == goal: break

        for q in G[p]:
            new_g = g[p] + G[p][q]
            if q in CLOSED: continue

            if g.get(q) is None:
                g[q] = new_g
                f[q] = g[q] + h.get(q, 0)
                path[q] = p
                OPEN.push((f[q], q))
            elif new_g < g[q]:
                old_item = (f[q], q)
                g[q] = new_g
                f[q] = g[q] + h.get(q, 0)
                path[q] = p
                OPEN.update(old_item, (f[q], q))
    return (path, g)

In [5]:
print("===== 1. BEST FIRST SEARCH (DIJKSTRA) =====")
path1, g1 = BestFirstSearch(COST, 'A', 'B')
print(f"Đường đi: {' -> '.join(find_path(path1, 'A', 'B'))}")
print(f"Tổng chi phí: {g1.get('B')}")

print("\n===== 2. A* SEARCH =====")
path2, g2 = AStartSearch(COST, 'A', 'B', h)
print(f"Đường đi: {' -> '.join(find_path(path2, 'A', 'B'))}")
print(f"Tổng chi phí: {g2.get('B')}")

===== 1. BEST FIRST SEARCH (DIJKSTRA) =====
Đường đi: A -> D -> E -> I -> B
Tổng chi phí: 19

===== 2. A* SEARCH =====
Đường đi: A -> D -> E -> I -> B
Tổng chi phí: 19


# Bài 2: Tìm đường đi ngắn nhất từ 2 đỉnh trong đồ thị với tri thức cho trước

## 1. Giới thiệu

OPEN: tập các trạng thái đã được sinh ra nhưng chưa được xét đến.
CLOSED: tập các trạng thái đã được xét đến.
COST(p, q): là khoảng cách giữa p, q.
g(p): khoảng cách từ trạng thái đầu đến trạng thái hiện tại p.
path(p): trạng thái trước đó để đi tới trạng thái p
Bước 1:
Open = {s}
Close = {}
Bước 2: while (Open !={})
+ Chọn trạng thái (đỉnh) tốt nhất p theo g(p) trong OPEN (xóa p khỏi OPEN).
+ Chuyển p qua CLOSED
+ Xét các đỉnh q kề với p:
* Nếu q chưa có trong OPEN và CLOSED:
- g(q) = g(p) + cost(p, q)
- path(q) = p
- Thêm q vào OPEN
* Nếu q đã có trong OPEN và g(q) > g(p) + cost(p, q):
- g(q) = g(p) + cost(p, q)
- path(q) = p
* Nếu q có trong CLOSED: không làm gì cả
Bước 3: Truy hồi trạng thái đích.

---
## 2. Bài toán

Tìm đường đi ngắn nhất từ 2 đỉnh trong đồ thị với tri thức cho trước




In [6]:
import heapq
import pprint

class MinHeap:
    def __init__(self):
        self.items = []

    def empty(self):
        return len(self.items) == 0

    def push(self, item):
        heapq.heappush(self.items, item)

    def pop(self):
        return heapq.heappop(self.items)

    def check(self, item):
        return item in self.items

In [7]:
# Biểu diễn đồ thị có trọng số
COST = {
    'A': {'C': 9, 'D': 7, 'E': 13, 'F': 20},
    'B': {},
    'C': {'H': 6},
    'D': {'E': 4, 'H': 8},
    'E': {'I': 3, 'K': 4},
    'F': {'G': 4, 'I': 6},
    'G': {},
    'H': {'K': 5},
    'I': {'B': 5, 'K': 9},
    'K': {'B': 6}
}

# Giá trị Heuristic h(n)
h = {
    'A': 14, 'B': 0, 'C': 15, 'D': 6, 'E': 8,
    'F': 7, 'G': 12, 'H': 10, 'I': 4, 'K': 2
}

def LayKe(G, a):
    if G.get(a) is None:
        return None
    return list(G[a].keys())

In [8]:
def AStarSearch(G, start, goal, h):
    if G.get(start) is None or G.get(goal) is None:
        return (None, None)

    path = {}   # lưu đỉnh trước để truy hồi
    g = {}      # chi phí thực tế từ điểm bắt đầu
    f = {}      # tổng chi phí dự đoán f = g + h

    OPEN = MinHeap()
    CLOSED = []

    path[start] = None
    g[start] = 0
    f[start] = h.get(start, 0)

    OPEN.push((f[start], start))

    while not OPEN.empty():
        (_, p) = OPEN.pop()
        
        if p in CLOSED:
            continue
            
        CLOSED.append(p)

        if p == goal:
            break

        ds_ke = LayKe(G, p)
        if ds_ke is None: continue

        for q in ds_ke:
            cost_pq = G[p][q]
            new_g = g[p] + cost_pq

            # Kiểm tra và cập nhật chi phí
            if q in CLOSED:
                if g[q] > new_g:
                    g[q] = new_g
                    f[q] = g[q] + h.get(q, 0)
                    path[q] = p
                    CLOSED.remove(q)
                    OPEN.push((f[q], q))
            elif q in g:
                if g[q] > new_g:
                    g[q] = new_g
                    f[q] = g[q] + h.get(q, 0)
                    path[q] = p
                    OPEN.push((f[q], q))
            else:
                g[q] = new_g
                f[q] = g[q] + h.get(q, 0)
                path[q] = p
                OPEN.push((f[q], q))

    return (path, g)

In [9]:
def find_path(path, start, goal):
    result = []
    if goal not in path:
        return []

    v = goal
    while v is not None:
        result.append(v)
        v = path[v]

    result.reverse()
    return result

In [10]:
# Thực thi thuật toán
path_result, g_result = AStarSearch(COST, 'A', 'B', h)

print("-" * 30)
print("KẾT QUẢ TÌM KIẾM A*")
print("-" * 30)

print("\n1. Mảng chi phí g(n):")
pprint.pprint(g_result)

print("\n2. Mảng truy hồi (Node cha):")
pprint.pprint(path_result)

duong_di = find_path(path_result, 'A', 'B')

print("\n3. Kết quả cuối cùng:")
if duong_di:
    print(f"   * Đường đi: {' -> '.join(duong_di)}")
    print(f"   * Tổng chi phí: {g_result['B']}")
else:
    print("   * Không tìm thấy đường đi!")

------------------------------
KẾT QUẢ TÌM KIẾM A*
------------------------------

1. Mảng chi phí g(n):
{'A': 0, 'B': 19, 'C': 9, 'D': 7, 'E': 11, 'F': 20, 'H': 15, 'I': 14, 'K': 15}

2. Mảng truy hồi (Node cha):
{'A': None,
 'B': 'I',
 'C': 'A',
 'D': 'A',
 'E': 'D',
 'F': 'A',
 'H': 'D',
 'I': 'E',
 'K': 'E'}

3. Kết quả cuối cùng:
   * Đường đi: A -> D -> E -> I -> B
   * Tổng chi phí: 19
